<a href="https://colab.research.google.com/github/dossamatheo/Banque-online/blob/main/q5_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

if not os.path.exists("client_propre.csv"):
    from google.colab import files
    files.upload()

df = pd.read_csv("client_propre.csv")
y = (df["est_revenu_6m"] == 0)

print("Nombre de clients :", len(df))
print("Part de clients qui ne reviennent pas (True) :", round(y.mean(), 3))
print("Part de clients qui reviennent (False)       :", round((~y).mean(), 3))
df.head()

Saving client_propre.csv to client_propre.csv
Nombre de clients : 679
Part de clients qui ne reviennent pas (True) : 0.591
Part de clients qui reviennent (False)       : 0.409


,client_id,boutique_principale,anciennete_mois,nb_achats_12m,panier_moyen_eur,retours,demandes_sav,carte_fidelite,canal_prefere,pays_residence,montant_prochain_achat_eur,est_revenu_6m,montant_total_eur
0,CT80059,BT15,4,4,1468.0,0,0,aucune,en ligne,usa,725.0,1,5872.0
1,CT80578,BT28,80,3,1235.0,1,1,argent,boutique,france,1479.0,1,3705.0
2,CT80256,BT27,43,1,1814.0,0,2,or,boutique,france,2307.0,1,1814.0
3,CT80098,BT16,1,1,1795.0,0,0,or,boutique,italie,1820.0,0,1795.0
4,CT80362,BT31,51,3,2022.0,0,0,argent,boutique,chine,1201.0,1,6066.0


In [2]:
colonnes = [
    "anciennete_mois",
    "nb_achats_12m",
    "panier_moyen_eur",
    "montant_total_eur",
    "retours",
    "demandes_sav",
]

print("=== Moyennes : partis (0) vs revenus (1) ===")
print(df.groupby("est_revenu_6m")[colonnes].mean().T.round(2))

print("\n=== Corrélations entre colonnes ===")
print(df[colonnes].corr().round(2))

=== Moyennes : partis (0) vs revenus (1) ===
est_revenu_6m            0         1
anciennete_mois      28.50     34.27
nb_achats_12m         3.15      4.39
panier_moyen_eur   2437.84   2938.97
montant_total_eur  9689.42  16916.82
retours               0.49      0.48
demandes_sav          0.40      0.40

=== Corrélations entre colonnes ===
                   anciennete_mois  nb_achats_12m  panier_moyen_eur  \
anciennete_mois               1.00          -0.05             -0.32   
nb_achats_12m                -0.05           1.00              0.54   
panier_moyen_eur             -0.32           0.54              1.00   
montant_total_eur            -0.21           0.84              0.85   
retours                      -0.12           0.30              0.31   
demandes_sav                  0.03          -0.05             -0.08   

                   montant_total_eur  retours  demandes_sav  
anciennete_mois                -0.21    -0.12          0.03  
nb_achats_12m                   0.84 

In [3]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

X = df[["anciennete_mois", "nb_achats_12m"]]
cv = StratifiedKFold(5, shuffle=True, random_state=42)

# pas standardiser (k = 21)
score_brut = cross_val_score(KNeighborsClassifier(21), X, y, cv=cv)

#standardisé (k = 21)
modele_std = make_pipeline(StandardScaler(), KNeighborsClassifier(21))
score_std = cross_val_score(modele_std, X, y, cv=cv)

print("Sans standardiser :", round(score_brut.mean(), 3))
print("Avec StandardScaler :", round(score_std.mean(), 3))

Sans standardiser : 0.583
Avec StandardScaler : 0.629


In [4]:
liste_k = [1, 3, 5, 7, 11, 15, 21, 31, 51]
justesses = []

for k in liste_k:
    modele = make_pipeline(StandardScaler(), KNeighborsClassifier(k))
    score = cross_val_score(modele, X, y, cv=cv)
    justesses.append(score.mean())
    print(f"k = {k:2d} | justesse = {score.mean():.3f} | ecart-type = {score.std():.3f}")

k =  1 | justesse = 0.530 | ecart-type = 0.030
k =  3 | justesse = 0.574 | ecart-type = 0.027
k =  5 | justesse = 0.573 | ecart-type = 0.024
k =  7 | justesse = 0.598 | ecart-type = 0.033
k = 11 | justesse = 0.611 | ecart-type = 0.038
k = 15 | justesse = 0.611 | ecart-type = 0.022
k = 21 | justesse = 0.629 | ecart-type = 0.031
k = 31 | justesse = 0.611 | ecart-type = 0.029
k = 51 | justesse = 0.613 | ecart-type = 0.041


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(liste_k, justesses, marker="o", label="Justesse moyenne (5 tours)")
plt.axhline(y.mean(), color="red", linestyle="--", label="Classe majoritaire (0,591)")
plt.xlabel("Nombre de voisins k")
plt.ylabel("Justesse moyenne")
plt.title("Choix de k en validation croisée (ancienneté et nb d'achats)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
revenus = df[df["est_revenu_6m"] == 1]
partis = df[df["est_revenu_6m"] == 0]

plt.figure(figsize=(8, 5))
plt.scatter(revenus["anciennete_mois"], revenus["nb_achats_12m"],
            color="green", alpha=0.5, label="Est revenu")
plt.scatter(partis["anciennete_mois"], partis["nb_achats_12m"],
            color="blue", alpha=0.5, label="N'est pas revenu")
plt.xlabel("Ancienneté (en mois)")
plt.ylabel("Nombre d'achats sur 12 mois")
plt.title("Répartition des clients selon l'ancienneté et la fréquence d'achat")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Standardisation : fit_transform sur l'apprentissage, transform seul sur le test
scaler = StandardScaler()
X_tr_std = scaler.fit_transform(X_tr)
X_te_std = scaler.transform(X_te)

knn = KNeighborsClassifier(21).fit(X_tr_std, y_tr)
pred = knn.predict(X_te_std)

exactitude = (pred == y_te).mean()
detections = (pred & y_te).sum()
fausses_alertes = (pred & (~y_te)).sum()

print("Exactitude sur le test         :", round(exactitude, 3))
print("Vrais départs détectés         :", detections, "sur", y_te.sum())
print("Clients fidèles appelés à tort :", fausses_alertes, "sur", (~y_te).sum())

In [ ]:
modele_final = make_pipeline(StandardScaler(), KNeighborsClassifier(21)).fit(X, y)

nouveaux = pd.DataFrame([
    [3, 1],    # Client récent, 1 seul achat
    [36, 4],   # Client intermédiaire
    [72, 10],  # Client ancien et très fidèle
], columns=["anciennete_mois", "nb_achats_12m"])

nouveaux["va_partir"] = modele_final.predict(nouveaux)
nouveaux["proba_depart"] = modele_final.predict_proba(nouveaux[["anciennete_mois", "nb_achats_12m"]])[:, 1].round(2)
nouveaux["voisins_partis_sur_21"] = (nouveaux["proba_depart"] * 21).round(0).astype(int)

print(nouveaux)